# 13 — Multimodal Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
from groq import Groq
client = Groq()
VISION_MODEL = "llama-3.2-11b-vision-preview"
TEXT_MODEL   = "llama-3.1-8b-instant"
print(f"Vision model : {VISION_MODEL}")
print(f"Text model   : {TEXT_MODEL}")


Vision model : llama-3.2-11b-vision-preview
Text model   : llama-3.1-8b-instant


## Vision API: Image URL Prompting

In [3]:
def vision_chat(user_text, image_url=None, model=VISION_MODEL, max_tokens=300):
    content = []
    if image_url:
        content.append({"type":"image_url","image_url":{"url":image_url}})
    content.append({"type":"text","text":user_text})
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":content}],
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content

test_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
try:
    result = vision_chat("Describe what you see in this image.", image_url=test_url)
    print("Image description:")
    print(result.strip())
except Exception as e:
    print(f"Vision API: {e.__class__.__name__}: {e}")
    print("(Vision model may require specific API tier)")


Vision API: BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama-3.2-11b-vision-preview` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
(Vision model may require specific API tier)


## Multimodal Prompt Strategies

In [4]:
strategies = [
    "Describe what you see in this image.",
    "What is the main subject of this image? Answer in one sentence.",
    "List 5 specific details visible in this image.",
    "What text, if any, appears in this image?",
    "Describe the mood or atmosphere of this image.",
]

test_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
for strategy in strategies:
    try:
        r = vision_chat(strategy, image_url=test_url, max_tokens=80)
        print(f"Q: {strategy[:50]}")
        print(f"A: {r.strip()[:100]}")
    except Exception as e:
        print(f"Q: {strategy[:50]}")
        print(f"A: [Vision API not available: {e.__class__.__name__}]")
    print()


Q: Describe what you see in this image.
A: [Vision API not available: BadRequestError]

Q: What is the main subject of this image? Answer in 
A: [Vision API not available: BadRequestError]

Q: List 5 specific details visible in this image.
A: [Vision API not available: BadRequestError]

Q: What text, if any, appears in this image?
A: [Vision API not available: BadRequestError]

Q: Describe the mood or atmosphere of this image.
A: [Vision API not available: BadRequestError]



## Text-Based Multimodal Simulation

In [5]:
# When vision API is unavailable, demonstrate with text descriptions

def chat_text(prompt, max_tokens=200):
    resp = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{"role":"user","content":prompt}],
        max_tokens=max_tokens, temperature=0
    )
    return resp.choices[0].message.content

image_description = (
    "[Image description]: A bar chart showing monthly sales data for Q1 2024.\n"
    "January: $120,000 | February: $95,000 | March: $140,000.\n"
    "Title: Q1 Sales Performance. Y-axis: Revenue in USD. X-axis: Months."
)

analysis_prompt = (
    f"Based on this chart description, provide:\n"
    f"1. Key insight (one sentence)\n"
    f"2. Best/worst month\n"
    f"3. Trend observation\n"
    f"4. One recommendation\n\n"
    f"Chart: {image_description}"
)
print(chat_text(analysis_prompt))


Based on the provided chart description, here are the requested insights:

1. **Key insight**: Sales performance in Q1 2024 showed a fluctuating trend, with January and March experiencing significant growth, while February saw a decline.
2. **Best month**: March, with sales of $140,000.
3. **Trend observation**: The sales data exhibits a seasonal or irregular pattern, with no clear upward or downward trend throughout the quarter.
4. **Recommendation**: To improve sales consistency and potentially capitalize on the growth seen in January and March, consider analyzing the factors that contributed to these increases and exploring ways to replicate them in other months.


## Multimodal Prompt Best Practices

In [6]:
best_practices = {
    "Be specific":     "Instead of 'describe this', ask 'list the colors and objects visible'",
    "Specify format":  "Ask for bullet points, JSON, or specific structure",
    "Chain tasks":     "First describe, then analyze, then extract specific info",
    "Grounding":       "Reference specific regions: 'in the top-right corner'",
    "Comparison":      "When given multiple images: 'comparing image 1 and 2'",
}
print("Multimodal Prompting Best Practices:")
print()
for tip, example in best_practices.items():
    print(f"  {tip:15s}: {example}")


Multimodal Prompting Best Practices:

  Be specific    : Instead of 'describe this', ask 'list the colors and objects visible'
  Specify format : Ask for bullet points, JSON, or specific structure
  Chain tasks    : First describe, then analyze, then extract specific info
  Grounding      : Reference specific regions: 'in the top-right corner'
  Comparison     : When given multiple images: 'comparing image 1 and 2'
